In [26]:
import sys, os, random
sys.path.append(os.path.join(os.getcwd(), 'CONFOLD')) #add CONFOLD to path

import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

import pandas as pd

from foldrm import Classifier
from datasets import  final_extinctionrisk_dataframe, final_extinctionrisk_noth_dataframe

from algo import prune_rules

In [27]:
random.seed(42)

# With Human Threats

In [28]:
model_template, data = final_extinctionrisk_dataframe()
X = data[model_template.attrs].drop('Extinction_risk',axis=1)
X = X.convert_dtypes()
X = X.to_numpy()
y = data['Extinction_risk'].to_numpy()

# Loading the Train/Test Splits from R

In [29]:
import csv

train_indices = []
test_indices = []
for i in range(1,11):
    with open("./datasets/r_folds/fold_"+str(i)+"_train_idx.csv", newline='') as csvfile:
        train_df = pd.read_csv(csvfile)
        train_list = train_df.to_numpy().flatten().tolist()
        train_indices.append(train_list)
    with open("./datasets/r_folds/fold_"+str(i)+"_test_idx.csv", newline='') as csvfile:
        test_df = pd.read_csv(csvfile)
        test_list = train_df.to_numpy().flatten().tolist()
        test_indices.append(test_list)

# K-Fold Cross Validation

In [ ]:
acc_metrics = []
f1_metrics = [] 
precision_metrics = []
recall_metrics = []


for i, (train_index, test_index) in enumerate(zip(train_indices, test_indices)):
    print("Fold "+str(i)+":")
    temp_x_train = X[train_index]
    baseline_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
    train_data = np.concatenate((np.array(X[train_index]), np.array(y[train_index]).reshape(-1, 1)), axis=1).tolist()
    test_data = np.concatenate((np.array(X[test_index]), np.array(y[test_index]).reshape(-1, 1)), axis=1).tolist()

    baseline_model.fit(train_data, ratio=0.5)
    # Prepare the test data (features and true labels)
    X_test = [d[:-1] for d in test_data]
    Y_test = [d[-1] for d in test_data]

    # Get predictions (these will be tuples of (label, confidence))
    predictions_tuples = baseline_model.predict(X_test)
    predicted_labels = [p[0] for p in predictions_tuples]
    accuracy = accuracy_score(Y_test, predicted_labels)


    # Instantiate a new classifier for our expert-guided model
    expert_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)

    # Now, fit the model on the training data.
    # The algorithm will work around the rules we provided.
    expert_model.fit(train_data, ratio=0.75)

    # Get predictions from our new model
    expert_predictions_tuples = expert_model.predict(X_test)
    expert_predicted_labels = [p[0] for p in expert_predictions_tuples]
    expert_accuracy = accuracy_score(Y_test, expert_predicted_labels)


    
    print("--- Baseline Model Evaluation ---")
    print(f"Accuracy: {accuracy * 100:.2f}%\n")

    print("--- Expert Model Evaluation ---")
    print(f"Accuracy: {expert_accuracy * 100:.2f}%")

    # Instantiate a new classifier
    learned_confidence_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)


    # Now, fit the model on the training data.
    # The algorithm will calculate the confidence of our provided rules and then learn any additional rules needed.
    learned_confidence_model.fit(train_data, ratio=0.5)


    # Get predictions from our new model
    learned_conf_predictions = learned_confidence_model.predict(X_test)
    learned_conf_labels = [p[0] for p in learned_conf_predictions]

    # Calculate accuracy
    learned_conf_accuracy = accuracy_score(Y_test, learned_conf_labels)

    print("--- Learned Confidence Model Evaluation ---")
    #print(f"True Labels:      {Y_test}")
    #print(f"Predicted Labels: {learned_conf_labels}")
    print(f"Accuracy: {learned_conf_accuracy * 100:.2f}%")


    # Apply the pruning function
    # This will create a new list containing only the rules that meet the confidence threshold.
    pruned_rules = prune_rules(expert_model.rules, confidence=0.90)

    # We can create a new model instance to hold these pruned rules
    simple_pruned_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
    simple_pruned_model.rules = pruned_rules

    #print("\n--- Rules After Pruning (Confidence >= 0.90) ---")
    #simple_pruned_model.print_asp(simple=True)
                
    # Instantiate a new model for this experiment
    advanced_pruning_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)

    ##################
    #### Method 2: Advanced Confidence-Driven Learning

    # Now, train using confidence_fit with a high 15% improvement threshold
    #print("--- Training with confidence_fit(improvement_threshold=0.15) ---")
    #print("\n--- Rules Learned via Confidence-Driven Learning ---")
    #print("Note how the model is simpler and did not learn any exceptions to rules or `abnormalities', as they did not meet the high confidence improvement threshold.")
    advanced_pruning_model.confidence_fit(train_data, improvement_threshold=0.20)
    advanced_pruning_model.print_asp(simple=True)


    adv_pruning_predictions = advanced_pruning_model.predict(X_test)
    adv_pruning_labels = [p[0] for p in adv_pruning_predictions]

    acc = accuracy_score(Y_test, adv_pruning_labels)

    f1 = f1_score(Y_test, adv_pruning_labels, average=None, labels=["Higher_risk", "Lower_risk"]) #["Higher_risk", "Lower_risk"]["Lower_risk", "Higher_risk"]
    precision = precision_score(Y_test,adv_pruning_labels, average=None, labels=["Higher_risk", "Lower_risk"])
    recall = recall_score(Y_test, adv_pruning_labels, average=None, labels=["Higher_risk", "Lower_risk"])
    
    acc_metrics.append(acc)    
    f1_metrics.append(f1)
    precision_metrics.append(precision)
    recall_metrics.append(recall)

    print()
    print("----------------------------------------")
    print()

Fold 0:
--- Baseline Model Evaluation ---
Accuracy: 92.60%

--- Expert Model Evaluation ---
Accuracy: 92.94%
--- Learned Confidence Model Evaluation ---
Accuracy: 92.60%
extinction_risk(X,'lower_risk') :- hunting(X,N3), N3<=0. [confidence: 0.94723]
extinction_risk(X,'higher_risk') :- range_size(X,N25), N25<=11549.62. [confidence: 0.86842]
extinction_risk(X,'higher_risk') :- agriculture(X,N2), N2>0, range_size(X,N25), N25<=5118144.61. [confidence: 0.67252]
extinction_risk(X,'lower_risk') :- maximum_latitude(X,N13), N13>65.94. [confidence: 0.81496]
extinction_risk(X,'lower_risk') :- adult_survival_annual(X,N23), N23<=0.86675, range_size(X,N25), N25>6168290.72. [confidence: 0.80814]
extinction_risk(X,'higher_risk') :- invasive_species(X,N4), N4>0. [confidence: 0.67708]
extinction_risk(X,'lower_risk') :- tarsus_length(X,N8), N8<=91.6. [confidence: 0.65951]
extinction_risk(X,'higher_risk') :- clutch_size(X,N27), N27<=3.73214. [confidence: 0.78333]
extinction_risk(X,'lower_risk') :- minimum_

In [31]:
print(f"KFold Mean Acc: {np.mean(acc_metrics)} | KFold SD: {np.std(acc_metrics)}")
f1_metrics = np.array(f1_metrics)
precision_metrics = np.array(precision_metrics)
recall_metrics = np.array(recall_metrics)
print(f"KFold Mean f1: {np.mean(f1_metrics[:,0]),np.mean(f1_metrics[:,1])} | KFold SD: {np.std(f1_metrics[:,0]), np.std(f1_metrics[:,1])}")
print(f"KFold Mean Precision: {np.mean(precision_metrics[:, 0]), np.mean(precision_metrics[:, 1])} | KFold SD: {np.std(precision_metrics[:, 0]), np.std(precision_metrics[:, 1])}")
print(f"KFold Mean Recall: {np.mean(recall_metrics[:, 0]), np.mean(recall_metrics[:, 1])} | KFold SD: {np.std(recall_metrics[:, 0]), np.std(recall_metrics[:, 1])}")

KFold Mean Acc: 0.9076058307345771 | KFold SD: 0.0021905289070783367
KFold Mean f1: (np.float64(0.743381707733089), np.float64(0.9436556651640172)) | KFold SD: (np.float64(0.004328104656784196), np.float64(0.0015067336212820793))
KFold Mean Precision: (np.float64(0.7885378382147643), np.float64(0.9320998310407891)) | KFold SD: (np.float64(0.015461104452157383), np.float64(0.0026302344920057256))
KFold Mean Recall: (np.float64(0.7035759595160249), np.float64(0.9555313613805645)) | KFold SD: (np.float64(0.01354718971744472), np.float64(0.005049602945512613))


# Training on Complete Dataset withought Expert Rules and with Human Threats

In [32]:
baseline_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
train_data = np.concatenate((np.array(X), np.array(y).reshape(-1, 1)), axis=1).tolist()


baseline_model.fit(train_data, ratio=0.5)
expert_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
expert_model.fit(train_data, ratio=0.75)
learned_confidence_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
learned_confidence_model.fit(train_data, ratio=0.5)
pruned_rules = prune_rules(expert_model.rules, confidence=0.90)
simple_pruned_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
simple_pruned_model.rules = pruned_rules
advanced_pruning_model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
advanced_pruning_model.confidence_fit(train_data, improvement_threshold=0.20)
advanced_pruning_model.print_asp(simple=True)

extinction_risk(X,'lower_risk') :- agriculture(X,N2), N2<=0. [confidence: 0.94195]
extinction_risk(X,'higher_risk') :- range_size(X,N25), N25<=13551.2. [confidence: 0.87655]
extinction_risk(X,'higher_risk') :- not realm(X,'l'), range_size(X,N25), N25<=5136924.07. [confidence: 0.70704]
extinction_risk(X,'lower_risk') :- tail_length(X,N11), N11<=304.0, maximum_latitude(X,N13), N13>1.03, range_size(X,N25), N25<=19212039.47, body_mass(X,N26), N26<=79.2. [confidence: 0.77899]
extinction_risk(X,'lower_risk') :- range_size(X,N25), N25>19212039.47. [confidence: 0.89024]
extinction_risk(X,'higher_risk') :- tail_length(X,N11), N11>304.0. [confidence: 0.82258]
extinction_risk(X,'lower_risk') :- not realm(X,'p'), maximum_elevation(X,N22), N22>1600.0, adult_survival_annual(X,N23), N23<=0.87484, range_size(X,N25), N25>354187.36. [confidence: 0.75]
extinction_risk(X,'higher_risk') :- not realm(X,'p'), beak_depth(X,N7), N7>3.3, hand_wing_index(X,N10), N10>18.9. [confidence: 0.68155]
extinction_risk(X,